In [69]:
# QwenLog Real-Time Log Anomaly Detection Benchmark
# Comparing: Real-Time Parsing vs Pre-Parsed (DataLoader) Methods

import pandas as pd
import json
import sys
import re
import time
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

import torch
import torch.nn as nn

sys.path.insert(0, './training')

In [70]:
# Load configuration and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("./models/training_metadata_BGL.json", "r") as f:
    metadata = json.load(f)
with open("./models/event_vocab_BGL.json", "r") as f:
    event_vocab = json.load(f)

sequence_length = metadata['sequence_length']
block_size = metadata['block_size']

print(f"Device: {device}")
print(f"Sequence length: {sequence_length}, Block size: {block_size}, Vocab: {len(event_vocab)}")

Device: cuda
Sequence length: 80, Block size: 500, Vocab: 99


In [ ]:
# Model Architecture
class EventSequenceModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dropout=.3), num_layers=2)
        self.attn = nn.Linear(embed_dim, 1)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(.3), nn.Linear(hidden_dim, 2))

    def forward(self, x):
        batch_size, seq_len = x.size()
        x = self.embed(x)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = x + self.pos_encoder(positions)
        x = self.transformer(x.permute(1, 0, 2))
        attn_weights = torch.softmax(self.attn(x), dim=0)
        return self.fc((x * attn_weights).sum(dim=0))

model = EventSequenceModel(metadata['vocab_size'], metadata['embed_dim'], metadata['hidden_dim'])
model.load_state_dict(torch.load("./models/TableVIII_bgl_realtime_classifier.pth", weights_only=True))
model.to(device).eval()
print("✓ Model loaded")

✓ Model loaded


/home/sslab/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [72]:
# QwenLog Deterministic Real-Time Parser
import qwenlog_full_parser_bank as parser_bank

class QwenLogRealTimeParser:
    """Deterministic parser - sequential template matching (1→1848), first match wins"""
    
    def __init__(self, event_vocab):
        self.event_vocab = event_vocab
        self.unknown_idx = len(event_vocab) - 1
        self.all_parsers = []
        self.bgl_regex = re.compile(
            r'^(\S+)\s+(\d+)\s+(\d+\.\d+\.\d+)\s+(\S+)\s+(\d+-\d+-\d+-\d+\.\d+\.\d+\.\d+)\s+'
            r'(\S+)\s+(\w+)\s+(\w+)\s+(\w+)\s*(.*)$')
        self._load_parsers()
        print(f"✓ Loaded {len(self.all_parsers)} templates")
    
    def _load_parsers(self):
        for i in range(1, 2000):
            if hasattr(parser_bank, f'is_log_template_{i}'):
                self.all_parsers.append((i, getattr(parser_bank, f'is_log_template_{i}')))
    
    def parse_line(self, line):
        match = self.bgl_regex.match(line.strip())
        if not match:
            return None, None
        label = 0 if match.group(1) == '-' else 1
        content = match.group(10)
        
        for tid, is_func in self.all_parsers:
            try:
                if is_func(content):
                    return self.event_vocab.get(f"Q{tid}", self.unknown_idx), label
            except: pass
        return self.event_vocab.get("Q_UNKNOWN", self.unknown_idx), label

parser = QwenLogRealTimeParser(event_vocab)

✓ Loaded 1848 templates


In [73]:
# Real-Time Batched Pipeline
class RealTimePipeline:
    def __init__(self, parser, model, device, seq_len=80, batch_size=64):
        self.parser, self.model, self.device = parser, model, device
        self.seq_len, self.batch_size = seq_len, batch_size
        self.event_buf, self.label_buf = [], []
        self.pending_seqs, self.pending_labels = [], []
        self.all_preds, self.all_labels = [], []
        self.parse_time = self.classify_time = self.total_seqs = 0
    
    def process_line(self, line):
        t0 = time.perf_counter()
        event_idx, label = self.parser.parse_line(line)
        self.parse_time += time.perf_counter() - t0
        if event_idx is None: return
        
        self.event_buf.append(event_idx)
        self.label_buf.append(label)
        
        if len(self.event_buf) >= self.seq_len:
            self.pending_seqs.append(self.event_buf[:self.seq_len])
            self.pending_labels.append(1 if 1 in self.label_buf[:self.seq_len] else 0)
            self.total_seqs += 1
            self.event_buf.pop(0); self.label_buf.pop(0)
            if len(self.pending_seqs) >= self.batch_size:
                self._classify()
    
    def _classify(self):
        if not self.pending_seqs: return
        t0 = time.perf_counter()
        with torch.no_grad():
            x = torch.tensor(self.pending_seqs, dtype=torch.long, device=self.device)
            preds = self.model(x).argmax(dim=1).cpu().tolist()
        self.classify_time += time.perf_counter() - t0
        self.all_preds.extend(preds)
        self.all_labels.extend(self.pending_labels)
        self.pending_seqs, self.pending_labels = [], []
    
    def flush(self): self._classify()
    def reset_block(self): self.event_buf, self.label_buf = [], []

pipeline = RealTimePipeline(parser, model, device, sequence_length)
print("✓ Pipeline ready")

✓ Pipeline ready


In [74]:
# Setup test data
BGL_LOG_PATH = "./dataset/BGL.log"
total_lines = sum(1 for _ in open(BGL_LOG_PATH, 'r'))
test_start_line = int(total_lines * 0.8)
print(f"Total: {total_lines:,} | Test: {total_lines - test_start_line:,} logs (20%)")

Total: 4,747,963 | Test: 949,593 logs (20%)


In [75]:
# Run Real-Time Anomaly Detection
print("=" * 60)
print("REAL-TIME ANOMALY DETECTION (QwenLog Parser + Classifier)")
print("=" * 60)

start_time = time.time()
with open(BGL_LOG_PATH, 'r') as f:
    for line_num, line in enumerate(tqdm(f, total=total_lines, desc="Processing")):
        if line_num < test_start_line: continue
        if (line_num - test_start_line) % block_size == 0:
            pipeline.reset_block()
        pipeline.process_line(line)
pipeline.flush()
total_time = time.time() - start_time

print(f"\n✓ Sequences: {pipeline.total_seqs:,} | Time: {total_time:.2f}s")

REAL-TIME ANOMALY DETECTION (QwenLog Parser + Classifier)


Processing: 100%|██████████████████████████████████| 4747963/4747963 [00:32<00:00, 147952.70it/s]


✓ Sequences: 799,487 | Time: 32.09s


In [76]:
# Real-Time Results
cm = confusion_matrix(pipeline.all_labels, pipeline.all_preds)
TN, FP, FN, TP = cm.ravel()
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("=" * 60)
print("REAL-TIME RESULTS")
print("=" * 60)
print(f"Parse: {pipeline.parse_time:.2f}s | Classify: {pipeline.classify_time:.2f}s | Total: {total_time:.2f}s")
print(f"Throughput: {pipeline.total_seqs/total_time:,.0f} seq/s")
print(f"\nAccuracy: {accuracy*100:.2f}% | Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}% | F1: {f1_score*100:.2f}%")
print(f"\nConfusion: TP={TP:,} TN={TN:,} FP={FP:,} FN={FN:,}")

REAL-TIME RESULTS
Parse: 3.95s | Classify: 25.24s | Total: 32.09s
Throughput: 24,911 seq/s

Accuracy: 99.73% | Precision: 99.06%
Recall: 97.65% | F1: 98.35%

Confusion: TP=63,154 TN=734,209 FP=601 FN=1,523


In [77]:
# Pre-Parsed Method (DataLoader - Like Colleague's Approach)
print("=" * 60)
print("PRE-PARSED METHOD (No Real-Time Parsing)")
print("=" * 60)

df = pd.read_csv("labeled_logs_BGL_deterministic.csv")
df["EventIdx"] = df["EventId"].map(event_vocab)

# Same train/test split as training
block_starts = list(range(0, len(df), block_size))
block_labels = [1 if df.iloc[i:i+block_size]["Label"].sum() > 0 else 0 for i in block_starts]
_, test_blocks = train_test_split(block_starts, test_size=0.2, stratify=block_labels, random_state=42)

# Build test sequences
test_seqs, test_labels = [], []
for start in test_blocks:
    block = df.iloc[start:start + block_size].reset_index(drop=True)
    for i in range(len(block) - sequence_length):
        test_seqs.append(block["EventIdx"].iloc[i:i + sequence_length].tolist())
        test_labels.append(1 if 1 in block["Label"].iloc[i:i + sequence_length].tolist() else 0)

loader = DataLoader(TensorDataset(torch.tensor(test_seqs), torch.tensor(test_labels)), batch_size=64)
print(f"✓ Test sequences: {len(test_seqs):,}")

PRE-PARSED METHOD (No Real-Time Parsing)
✓ Test sequences: 798,000
✓ Test sequences: 798,000


In [78]:
# Run Pre-Parsed Inference
pp_preds, pp_labels = [], []
pp_start = time.time()
with torch.no_grad():
    for batch_x, batch_y in tqdm(loader, desc="Pre-parsed"):
        preds = model(batch_x.to(device)).argmax(dim=1).cpu().tolist()
        pp_preds.extend(preds)
        pp_labels.extend(batch_y.tolist())
pp_time = time.time() - pp_start
print(f"✓ Time: {pp_time:.2f}s")

Pre-parsed: 100%|█████████████████████████████████████████| 12469/12469 [00:27<00:00, 447.92it/s]

✓ Time: 27.84s


In [79]:
# Pre-Parsed Results
cm_pp = confusion_matrix(pp_labels, pp_preds)
TN_pp, FP_pp, FN_pp, TP_pp = cm_pp.ravel()
acc_pp = (TP_pp + TN_pp) / (TP_pp + TN_pp + FP_pp + FN_pp)
prec_pp = TP_pp / (TP_pp + FP_pp) if (TP_pp + FP_pp) > 0 else 0
rec_pp = TP_pp / (TP_pp + FN_pp) if (TP_pp + FN_pp) > 0 else 0
f1_pp = 2 * prec_pp * rec_pp / (prec_pp + rec_pp) if (prec_pp + rec_pp) > 0 else 0

print("\n" + "=" * 70)
print("                    COMPARISON: REAL-TIME vs PRE-PARSED")
print("=" * 70)
print(f"{'Metric':<25} {'Real-Time':<20} {'Pre-Parsed':<20}")
print("-" * 70)
print(f"{'Parsing':<25} {'On-the-fly':<20} {'None':<20}")
print(f"{'Total Time (s)':<25} {total_time:<20.2f} {pp_time:<20.2f}")
print(f"{'Throughput (seq/s)':<25} {pipeline.total_seqs/total_time:<20,.0f} {len(pp_preds)/pp_time:<20,.0f}")
print(f"{'Accuracy (%)':<25} {accuracy*100:<20.2f} {acc_pp*100:<20.2f}")
print(f"{'Precision (%)':<25} {precision*100:<20.2f} {prec_pp*100:<20.2f}")
print(f"{'Recall (%)':<25} {recall*100:<20.2f} {rec_pp*100:<20.2f}")
print(f"{'F1 Score (%)':<25} {f1_score*100:<20.2f} {f1_pp*100:<20.2f}")
print("=" * 70)


                    COMPARISON: REAL-TIME vs PRE-PARSED
Metric                    Real-Time            Pre-Parsed          
----------------------------------------------------------------------
Parsing                   On-the-fly           None                
Total Time (s)            32.09                27.84               
Throughput (seq/s)        24,911               28,665              
Accuracy (%)              99.73                99.86               
Precision (%)             99.06                98.98               
Recall (%)                97.65                99.60               
F1 Score (%)              98.35                99.29               


In [80]:
# Save Results
results = {
    "real_time": {
        "total_time_sec": round(total_time, 2),
        "parse_time_sec": round(pipeline.parse_time, 2),
        "throughput_seq_sec": round(pipeline.total_seqs/total_time),
        "accuracy": round(accuracy, 4), "precision": round(precision, 4),
        "recall": round(recall, 4), "f1_score": round(f1_score, 4),
        "TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN)
    },
    "pre_parsed": {
        "total_time_sec": round(pp_time, 2),
        "throughput_seq_sec": round(len(pp_preds)/pp_time),
        "accuracy": round(acc_pp, 4), "precision": round(prec_pp, 4),
        "recall": round(rec_pp, 4), "f1_score": round(f1_pp, 4),
        "TP": int(TP_pp), "TN": int(TN_pp), "FP": int(FP_pp), "FN": int(FN_pp)
    }
}

with open("./results/qwenlog_comparison_realtime_vs_preparsed.json", 'w') as f:
    json.dump(results, f, indent=2)

pd.DataFrame([
    {"Method": "Real-Time", "Time_sec": total_time, "Throughput": pipeline.total_seqs/total_time, 
     "Accuracy": accuracy, "F1": f1_score},
    {"Method": "Pre-Parsed", "Time_sec": pp_time, "Throughput": len(pp_preds)/pp_time,
     "Accuracy": acc_pp, "F1": f1_pp}
]).to_csv("./results/qwenlog_comparison_realtime_vs_preparsed.csv", index=False)

print("✓ Results saved to ./results/")

✓ Results saved to ./results/
